[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/19_cadenas_de_markov/notebooks/01_cadenas_y_simulacion.ipynb)

# Cadenas de Markov: Simulación y Matrices

**Módulo 19 — Cadenas de Markov · Notebook 01**

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

COLORS = {
    "blue": "#2E86AB", "red": "#E94F37", "green": "#27AE60",
    "gray": "#7F8C8D", "orange": "#F39C12", "purple": "#8E44AD",
    "dark": "#2C3E50", "teal": "#1ABC9C", "light": "#ECF0F1",
}

np.random.seed(42)

## ¿Qué haremos en este notebook?

1. **Construir** una clase `CadenaDeMarkov` que encapsule la simulación
2. **Analizar** texto español (El Quijote) clasificando vocales y consonantes — replicando el análisis original de Markov
3. **Modelar** regímenes de mercado (Bull/Bear/Flat) como cadena de 3 estados
4. **Explorar** potencias de la matriz $P^n$ y su convergencia
5. **Verificar** la distribución estacionaria empíricamente

## Parte 1 — La cadena de Markov como objeto

Una **cadena de Markov** (con espacio de estados finito) queda completamente definida por:

| Componente | Notación | Descripción |
|:---|:---:|:---|
| Estados | $S = \{s_1, \dots, s_k\}$ | Conjunto finito de estados posibles |
| Matriz de transición | $P \in \mathbb{R}^{k \times k}$ | $P_{ij} = \Pr(X_{t+1} = s_j \mid X_t = s_i)$ |
| Propiedad estocástica | $\sum_j P_{ij} = 1$ | Cada fila suma 1 |

La **propiedad de Markov** dice que el futuro solo depende del presente:

$$\Pr(X_{t+1} \mid X_t, X_{t-1}, \dots, X_0) = \Pr(X_{t+1} \mid X_t)$$

A continuación construimos una clase `CadenaDeMarkov` que:
- Valida que $P$ sea una matriz estocástica
- Ejecuta pasos individuales y trayectorias completas
- Calcula la distribución estacionaria vía eigenvectores

In [ ]:
class CadenaDeMarkov:
    """Cadena de Markov de estados finitos.

    Parametros
    ----------
    P : np.ndarray, shape (k, k)
        Matriz de transición (estocástica por filas).
    estados : list[str]
        Nombres legibles para cada estado.
    """

    def __init__(self, P, estados):
        P = np.asarray(P, dtype=float)
        k = P.shape[0]
        assert P.shape == (k, k), f"P debe ser cuadrada, recibí {P.shape}"
        sumas = P.sum(axis=1)
        assert np.allclose(sumas, 1.0), f"Filas no suman 1: {sumas}"
        assert len(estados) == k, "len(estados) debe coincidir con k"
        self.P = P
        self.k = k
        self.estados = list(estados)
        self._idx = {s: i for i, s in enumerate(estados)}

    # ── Un paso aleatorio ────────────────────────────────────────────────
    def paso(self, estado_actual):
        """Devuelve el siguiente estado (índice) dado el estado actual (índice)."""
        return np.random.choice(self.k, p=self.P[estado_actual])

    # ── Trayectoria completa ─────────────────────────────────────────────
    def simular(self, inicio, n_pasos, semilla=None):
        """Simula n_pasos a partir del estado `inicio` (índice o nombre).

        Retorna
        -------
        trayectoria : np.ndarray, shape (n_pasos + 1,)
            Arreglo de índices de estado visitados.
        """
        if semilla is not None:
            np.random.seed(semilla)
        if isinstance(inicio, str):
            inicio = self._idx[inicio]
        tray = np.empty(n_pasos + 1, dtype=int)
        tray[0] = inicio
        for t in range(n_pasos):
            tray[t + 1] = self.paso(tray[t])
        return tray

    # ── Distribución estacionaria (eigenvector izquierdo) ────────────────
    def distribucion_estacionaria(self):
        """Calcula π tal que πP = π.

        Usa el eigenvector izquierdo de P asociado al eigenvalor 1,
        equivalente al eigenvector derecho de P^T con eigenvalor 1.

        Retorna
        -------
        pi : np.ndarray, shape (k,)
        """
        vals, vecs = np.linalg.eig(self.P.T)
        # Buscar eigenvalor más cercano a 1
        idx = np.argmin(np.abs(vals - 1.0))
        pi = np.real(vecs[:, idx])
        pi = pi / pi.sum()          # normalizar
        return pi

    def __repr__(self):
        return f"CadenaDeMarkov(estados={self.estados}, k={self.k})"

print("Clase CadenaDeMarkov cargada ✓")

In [ ]:
# ── Prueba rápida: cadena Vocal / Consonante ────────────────────────
P_vc = np.array([
    [0.35, 0.65],   # V → V, V → C
    [0.52, 0.48],   # C → V, C → C
])
cadena_vc = CadenaDeMarkov(P_vc, estados=["V", "C"])
print(cadena_vc)

tray = cadena_vc.simular(inicio="V", n_pasos=10, semilla=7)
print("Trayectoria (10 pasos):", [cadena_vc.estados[i] for i in tray])

## Parte 2 — Vocales y Consonantes: el ejemplo original de Markov

En 1913, A. A. Markov publicó uno de los primeros análisis estadísticos de dependencia
secuencial en texto. Tomó los primeros 20 000 caracteres de *Eugene Onegin* de Pushkin,
clasificó cada letra como **vocal (V)** o **consonante (C)** y estimó la matriz de
transición $2 \times 2$.

Su hallazgo clave: las letras **no son independientes**. Las vocales tienden a ir seguidas
de consonantes y viceversa — la cadena tiene una fuerte tendencia a **alternar**.

Repetimos el experimento con el inicio de *Don Quijote de la Mancha* de Cervantes.

In [ ]:
# ── Texto: apertura de Don Quijote ────────────────────────────────────
texto_quijote = ("""En un lugar de la Mancha, de cuyo nombre no quiero acordarme, 
no ha mucho tiempo que vivía un hidalgo de los de lanza en astillero, adarga antigua, 
rocín flaco y galgo corredor. Una olla de algo más vaca que carnero, salpicón las más 
noches, duelos y quebrantos los sábados, lantejas los viernes, algún palomino de 
añadidura los domingos, consumían las tres partes de su hacienda. El resto della 
concluían sayo de velarte, calzas de velludo para las fiestas, con sus pantuflos de lo 
mesmo, y los días de entresemana se honraba con su vellorí de lo más fino. Tenía en su 
casa una ama que pasaba de los cuarenta, y una sobrina que no llegaba a los veinte, y un 
mozo de campo y plaza, que así ensillaba el rocín como tomaba la podadera. Frisaba la 
edad de nuestro hidalgo con los cincuenta años. Era de complexión recia, seco de carnes, 
enjuto de rostro, gran madrugador y amigo de la caza. Quieren decir que tenía el 
sobrenombre de Quijada, o Quesada, que en esto hay alguna diferencia en los autores que 
deste caso escriben, aunque por conjeturas verosímiles se deja entender que se llamaba 
Quijana. Pero esto importa poco a nuestro cuento: basta que en la narración dél no se 
salga un punto de la verdad.""")

# ── Limpieza ─────────────────────────────────────────────────────────

def limpiar(texto):
    """Minúsculas, quita acentos, conserva solo a-z."""
    texto = texto.lower()
    # Quitar acentos: á→a, é→e, etc.  ñ→n, ü→u
    reemplazos = {'á':'a','é':'e','í':'i','ó':'o','ú':'u','ü':'u','ñ':'n'}
    for orig, remp in reemplazos.items():
        texto = texto.replace(orig, remp)
    # Conservar solo a-z
    return ''.join(c for c in texto if 'a' <= c <= 'z')

limpio = limpiar(texto_quijote)
print(f"Caracteres limpios: {len(limpio)}")

# ── Clasificar V / C ────────────────────────────────────────────────
vocales = set('aeiou')
secuencia = ['V' if c in vocales else 'C' for c in limpio]
print(f"Primeros 60: {''.join(secuencia[:60])}")

# ── Contar transiciones ─────────────────────────────────────────────
conteo = np.zeros((2, 2))  # filas: V=0, C=1; columnas: V=0, C=1
mapa = {'V': 0, 'C': 1}

for i in range(len(secuencia) - 1):
    fila = mapa[secuencia[i]]
    col  = mapa[secuencia[i + 1]]
    conteo[fila, col] += 1

print(f"\nConteo de transiciones:")
print(f"  V→V={int(conteo[0,0])}  V→C={int(conteo[0,1])}")
print(f"  C→V={int(conteo[1,0])}  C→C={int(conteo[1,1])}")

# ── Normalizar → matriz de transición ───────────────────────────────
P_esp = conteo / conteo.sum(axis=1, keepdims=True)

print(f"\nMatriz de transición estimada (español):")
print(f"        V        C")
print(f"  V  [{P_esp[0,0]:.4f}   {P_esp[0,1]:.4f}]")
print(f"  C  [{P_esp[1,0]:.4f}   {P_esp[1,1]:.4f}]")

# Guardar frecuencias marginales
n_V = secuencia.count('V')
n_C = secuencia.count('C')
freq_V = n_V / len(secuencia)
freq_C = n_C / len(secuencia)
print(f"\nFrecuencia empírica: π_V ≈ {freq_V:.4f}, π_C ≈ {freq_C:.4f}")

### Comparación: Español (Quijote) vs. Ruso (Onegin, Markov 1913)

| Transición | Español (estimado) | Ruso (Markov 1913) |
|:---:|:---:|:---:|
| $P(V \to V)$ | *ver arriba* | 0.128 |
| $P(V \to C)$ | *ver arriba* | 0.872 |
| $P(C \to V)$ | *ver arriba* | 0.663 |
| $P(C \to C)$ | *ver arriba* | 0.337 |
| $\pi_V$ | *ver arriba* | ≈ 0.432 |
| $\pi_C$ | *ver arriba* | ≈ 0.568 |

**Observaciones:**
- Ambos idiomas muestran **fuerte alternancia** V↔C: la probabilidad de cambiar de clase es mayor que la de permanecer.
- El ruso tiene mayor alternancia ($P(V \to C) = 0.872$) que el español.
- Esto confirma el hallazgo de Markov: las letras en lenguaje natural **no son independientes** — la categoría actual influye fuertemente en la siguiente.

In [ ]:
# ── Simulación con la matriz estimada del español ────────────────────
cadena_esp = CadenaDeMarkov(P_esp, estados=["V", "C"])
pi_esp = cadena_esp.distribucion_estacionaria()
tray_esp = cadena_esp.simular(inicio=0, n_pasos=2000, semilla=42)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), gridspec_kw={"height_ratios": [1, 3]})

# ── Top: ribbon de estados (primeros 100 pasos) ─────────────────────
ax = axes[0]
n_show = 100
for t in range(n_show):
    color = COLORS["blue"] if tray_esp[t] == 0 else COLORS["red"]
    ax.axvspan(t, t + 1, color=color, alpha=0.8)
ax.set_xlim(0, n_show)
ax.set_yticks([])
ax.set_xlabel("Paso")
ax.set_title("Primeros 100 pasos de la cadena V/C (azul = V, rojo = C)")
patches = [mpatches.Patch(color=COLORS["blue"], label="V (vocal)"),
           mpatches.Patch(color=COLORS["red"], label="C (consonante)")]
ax.legend(handles=patches, loc="upper right", fontsize=9)

# ── Bottom: frecuencia acumulada de V ────────────────────────────────
ax = axes[1]
cum_V = np.cumsum(tray_esp == 0) / (np.arange(len(tray_esp)) + 1)
ax.plot(cum_V, color=COLORS["blue"], linewidth=1.2, label="Frecuencia acumulada de V")
ax.axhline(pi_esp[0], color=COLORS["dark"], linestyle="--", linewidth=1.5,
           label=f"π_V = {pi_esp[0]:.4f}")
ax.set_xlabel("Paso")
ax.set_ylabel("Frecuencia de V")
ax.set_title("Convergencia de la frecuencia empírica a π_V")
ax.legend(fontsize=10)
ax.set_ylim(0.2, 0.7)

plt.tight_layout()
plt.show()

## Parte 3 — Regímenes de Mercado

Los mercados financieros exhiben **regímenes** — periodos sostenidos con características estadísticas distintas:

| Régimen | Descripción | Rendimiento típico | Volatilidad |
|:---:|:---|:---:|:---:|
| **Bull** | Mercado alcista | $\mu > 0$ | Moderada |
| **Bear** | Mercado bajista | $\mu < 0$ | Alta |
| **Flat** | Mercado lateral | $\mu \approx 0$ | Baja |

Modelamos las transiciones entre estos regímenes como una cadena de Markov de 3 estados. Generamos datos sintéticos basados en parámetros históricos del S&P 500.

In [ ]:
# ── Generación de rendimientos mensuales con cambio de régimen ───────
# Datos basados en parámetros históricos del S&P 500.

np.random.seed(42)
n_months = 720
regime = 0
P_gen = np.array([
    [0.85, 0.08, 0.07],   # Bull → Bull, Bear, Flat
    [0.10, 0.75, 0.15],   # Bear → Bull, Bear, Flat
    [0.15, 0.10, 0.75],   # Flat → Bull, Bear, Flat
])
means = [0.015, -0.02, 0.003]
stds  = [0.03, 0.055, 0.025]

returns_sp500 = []
regimes_true = []
for t in range(n_months):
    r = np.random.normal(means[regime], stds[regime])
    returns_sp500.append(r)
    regimes_true.append(regime)
    regime = np.random.choice(3, p=P_gen[regime])

returns_sp500 = np.array(returns_sp500)
regimes_true = np.array(regimes_true)
years = np.linspace(1965, 2025, n_months)

print(f"Meses generados: {n_months}")
print(f"Rendimiento medio: {returns_sp500.mean():.4f}")
print(f"Volatilidad: {returns_sp500.std():.4f}")

> **Nota sobre el modelo generador vs. el modelo estimado**
>
> La matriz `P_gen` que usamos para generar los datos es el modelo "verdadero" — en la realidad, nunca lo conocemos. Lo que sí podemos hacer es **estimar** la matriz de transición a partir de los datos observados, clasificando retornos en regímenes con umbrales fijos. La matriz estimada diferirá de `P_gen` por dos razones: (1) la clasificación por umbrales es una aproximación cruda de los regímenes reales, y (2) con datos finitos siempre hay error de estimación. Esta brecha entre modelo verdadero y modelo estimado es fundamental en estadística — y motiva modelos más sofisticados como los **Modelos Ocultos de Markov** (HMMs).

In [ ]:
# ── Clasificar rendimientos en regímenes observados ──────────────────
labels_obs = np.where(returns_sp500 > 0.02, 0,         # Bull
             np.where(returns_sp500 < -0.02, 1, 2))    # Bear / Flat

nombres_reg = ["Bull", "Bear", "Flat"]
print("Distribución observada:")
for i, nombre in enumerate(nombres_reg):
    print(f"  {nombre}: {(labels_obs == i).sum()} meses ({(labels_obs == i).mean():.1%})")

# ── Contar transiciones y estimar P ─────────────────────────────────
conteo_3 = np.zeros((3, 3))
for t in range(len(labels_obs) - 1):
    conteo_3[labels_obs[t], labels_obs[t + 1]] += 1

P_mercado = conteo_3 / conteo_3.sum(axis=1, keepdims=True)

print(f"\nMatriz de transición estimada (3×3):")
print(f"{'':>8} {'Bull':>8} {'Bear':>8} {'Flat':>8}")
for i, nombre in enumerate(nombres_reg):
    print(f"  {nombre:>4}  [{P_mercado[i,0]:.4f}   {P_mercado[i,1]:.4f}   {P_mercado[i,2]:.4f}]")

In [ ]:
# ── Rendimientos mensuales coloreados por régimen observado ──────────
fig, ax = plt.subplots(figsize=(14, 5))

color_map = {0: COLORS["green"], 1: COLORS["red"], 2: COLORS["gray"]}
for t in range(len(returns_sp500)):
    ax.bar(years[t], returns_sp500[t], width=(years[1]-years[0]),
           color=color_map[labels_obs[t]], alpha=0.7, edgecolor="none")

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Año")
ax.set_ylabel("Rendimiento mensual")
ax.set_title("Rendimientos mensuales sintéticos — coloreados por régimen observado")

patches = [mpatches.Patch(color=COLORS["green"], label="Bull (>2%)"),
           mpatches.Patch(color=COLORS["red"],   label="Bear (<-2%)"),
           mpatches.Patch(color=COLORS["gray"],  label="Flat")]
ax.legend(handles=patches, loc="upper left", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Comparar duraciones de régimen: datos vs. simulación ─────────────
cadena_merc = CadenaDeMarkov(P_mercado, estados=nombres_reg)
tray_merc = cadena_merc.simular(inicio=0, n_pasos=n_months - 1, semilla=123)

def duraciones(seq):
    """Calcula la duración de cada racha consecutiva."""
    durs = {0: [], 1: [], 2: []}
    count = 1
    for t in range(1, len(seq)):
        if seq[t] == seq[t - 1]:
            count += 1
        else:
            durs[seq[t - 1]].append(count)
            count = 1
    durs[seq[-1]].append(count)
    return durs

durs_obs = duraciones(labels_obs)
durs_sim = duraciones(tray_merc)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for i, (nombre, color) in enumerate(zip(nombres_reg, ["green", "red", "gray"])):
    ax = axes[i]
    bins = np.arange(0.5, max(max(durs_obs[i], default=1), max(durs_sim[i], default=1)) + 1.5, 1)
    ax.hist(durs_obs[i], bins=bins, alpha=0.6, color=COLORS[color], label="Observado", density=True)
    ax.hist(durs_sim[i], bins=bins, alpha=0.4, color=COLORS["dark"], label="Simulado", density=True,
            histtype="step", linewidth=2)
    ax.set_title(f"Duración: {nombre}")
    ax.set_xlabel("Meses consecutivos")
    if i == 0:
        ax.set_ylabel("Densidad")
    ax.legend(fontsize=9)

plt.suptitle("Distribución de duración de regímenes: observado vs. simulado", fontsize=13)
plt.tight_layout()
plt.show()

## Parte 4 — Potencias de la Matriz: $P^n$

La entrada $(i, j)$ de $P^n$ tiene un significado directo:

$$P^n_{ij} = \Pr(X_{t+n} = j \mid X_t = i)$$

Es decir, la probabilidad de ir del estado $i$ al estado $j$ en **exactamente** $n$ pasos.

A medida que $n \to \infty$, las filas de $P^n$ convergen a la distribución estacionaria $\pi$ (para cadenas ergódicas). Esto significa que **la memoria del estado inicial se pierde**.

In [ ]:
# ── P^n para la cadena V/C: grilla de heatmaps ──────────────────────
ns = [1, 2, 4, 8, 16, 32]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for idx, n in enumerate(ns):
    ax = axes[idx // 3, idx % 3]
    Pn = np.linalg.matrix_power(P_esp, n)
    im = ax.imshow(Pn, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["V", "C"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["V", "C"])
    ax.set_title(f"$P^{{{n}}}$", fontsize=13)
    # Anotar valores
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{Pn[i,j]:.4f}", ha="center", va="center",
                    fontsize=11, color="white" if Pn[i,j] > 0.5 else "black")

plt.suptitle("Convergencia de $P^n$ — Cadena Vocal/Consonante", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Convergencia de P^n[i,j] hacia π ─────────────────────────────────
max_n = 40
p00 = []  # P^n[V→V]
p10 = []  # P^n[C→V]

for n in range(1, max_n + 1):
    Pn = np.linalg.matrix_power(P_esp, n)
    p00.append(Pn[0, 0])
    p10.append(Pn[1, 0])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, max_n + 1), p00, "o-", color=COLORS["blue"], markersize=4,
        label=r"$P^n_{V \to V}$ (empieza en V)")
ax.plot(range(1, max_n + 1), p10, "s-", color=COLORS["red"], markersize=4,
        label=r"$P^n_{C \to V}$ (empieza en C)")
ax.axhline(pi_esp[0], color=COLORS["dark"], linestyle="--", linewidth=2,
           label=f"π_V = {pi_esp[0]:.4f}")
ax.set_xlabel("n (número de pasos)")
ax.set_ylabel("Probabilidad de estar en V")
ax.set_title("Ambas curvas convergen a π_V — la memoria del inicio se pierde")
ax.legend(fontsize=11)
ax.set_ylim(0.3, 0.6)
plt.tight_layout()
plt.show()

## Parte 5 — Distribución Estacionaria

La distribución estacionaria $\pi$ satisface:

$$\pi P = \pi, \qquad \sum_i \pi_i = 1$$

**Interpretaciones equivalentes:**
- $\pi_i$ = fracción del tiempo que la cadena pasa en estado $i$ (a largo plazo)
- $\pi$ = la distribución que no cambia al aplicar un paso de la cadena
- Si $X_0 \sim \pi$, entonces $X_t \sim \pi$ para todo $t$

**Cálculo:** $\pi$ es el **eigenvector izquierdo** de $P$ con eigenvalor 1, normalizado para que sume 1. Equivale al eigenvector derecho de $P^T$.

In [ ]:
# ── Distribución estacionaria para ambas cadenas ─────────────────────
pi_vc = cadena_esp.distribucion_estacionaria()
pi_merc = cadena_merc.distribucion_estacionaria()

print("═" * 50)
print("Cadena Vocal/Consonante (español):")
print(f"  π = [{pi_vc[0]:.6f}, {pi_vc[1]:.6f}]")
print(f"  Verificación πP ≈ π: {np.allclose(pi_vc @ P_esp, pi_vc)}")

print()
print("Cadena de Regímenes de Mercado:")
for i, nombre in enumerate(nombres_reg):
    print(f"  π_{nombre} = {pi_merc[i]:.6f}")
print(f"  Verificación πP ≈ π: {np.allclose(pi_merc @ P_mercado, pi_merc)}")
print("═" * 50)

In [ ]:
# ── π teórica vs. frecuencias empíricas (10,000 pasos) ───────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── V/C chain ───────────────────────────────────────────────────────
ax = axes[0]
tray_long_vc = cadena_esp.simular(inicio=0, n_pasos=10_000, semilla=99)
freq_vc = np.array([(tray_long_vc == i).mean() for i in range(2)])

x = np.arange(2)
w = 0.35
ax.bar(x - w/2, pi_vc, w, color=COLORS["blue"], label="π (teórica)", alpha=0.85)
ax.bar(x + w/2, freq_vc, w, color=COLORS["orange"], label="Empírica (10k pasos)", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(["V", "C"])
ax.set_ylabel("Probabilidad")
ax.set_title("Vocal / Consonante")
ax.legend(fontsize=10)
for i in range(2):
    ax.text(i - w/2, pi_vc[i] + 0.01, f"{pi_vc[i]:.4f}", ha="center", fontsize=9)
    ax.text(i + w/2, freq_vc[i] + 0.01, f"{freq_vc[i]:.4f}", ha="center", fontsize=9)

# ── Market chain ─────────────────────────────────────────────────────
ax = axes[1]
tray_long_merc = cadena_merc.simular(inicio=0, n_pasos=10_000, semilla=99)
freq_merc = np.array([(tray_long_merc == i).mean() for i in range(3)])

x = np.arange(3)
ax.bar(x - w/2, pi_merc, w, color=COLORS["teal"], label="π (teórica)", alpha=0.85)
ax.bar(x + w/2, freq_merc, w, color=COLORS["orange"], label="Empírica (10k pasos)", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(nombres_reg)
ax.set_ylabel("Probabilidad")
ax.set_title("Regímenes de Mercado")
ax.legend(fontsize=10)
for i in range(3):
    ax.text(i - w/2, pi_merc[i] + 0.01, f"{pi_merc[i]:.4f}", ha="center", fontsize=9)
    ax.text(i + w/2, freq_merc[i] + 0.01, f"{freq_merc[i]:.4f}", ha="center", fontsize=9)

plt.suptitle("Distribución estacionaria: teórica vs. empírica", fontsize=14)
plt.tight_layout()
plt.show()

## Resumen

| Concepto | Qué aprendimos |
|:---|:---|
| **Matriz de transición** $P$ | Define completamente una cadena de Markov con estados finitos; cada fila suma 1 |
| **Propiedad de Markov** | El futuro depende solo del presente, no del pasado |
| **Simulación** | Con `np.random.choice(k, p=P[i])` generamos trayectorias paso a paso |
| **Vocales / Consonantes** | Markov (1913) demostró dependencia secuencial en texto; replicamos con el Quijote |
| **Regímenes de mercado** | Las cadenas de 3+ estados modelan fenómenos complejos como ciclos financieros |
| **Potencias** $P^n$ | $(P^n)_{ij}$ = probabilidad de ir de $i$ a $j$ en $n$ pasos; convergen a $\pi$ |
| **Distribución estacionaria** $\pi$ | Eigenvector izquierdo con eigenvalor 1; frecuencia de largo plazo por estado |
| **Convergencia empírica** | La frecuencia observada en simulaciones largas se aproxima a $\pi$ |

**Siguiente notebook:** MCMC — usaremos cadenas de Markov para muestrear distribuciones complejas.